# 04 — Lectures 1–34 End-to-End Product

## Website + Conversational Agent Memory + Quiz + Weak Topics + Course Builder + LangSmith

This notebook is the end-to-end **multi-lecture application** built on the shared
Lectures 1–34 RAG system from Notebook 2.

The same application architecture is intended to scale later from 5 lectures
to the full course without rewriting the product logic.


# Step 0: Setup


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
!pip install -q     langchain     langchain-core     langchain-chroma     langchain-huggingface     langchain-openai     langgraph     langsmith     sentence-transformers     gradio     pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.1/125.1 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 85.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 120.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 37.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.

In [ ]:
from pathlib import Path
import json
import os
import operator
import re
import uuid
from typing import Annotated, Literal
from typing_extensions import NotRequired

PROJECT_ROOT = Path("/content/drive/MyDrive/AI_Engineering_Final_Project")

RAG_CONFIG_PATH = (
    PROJECT_ROOT
    / "rag"
    / "all_lectures_rag_config.json"
)

OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

HTML_OUTPUT_PATH = (
    OUTPUT_DIR
    / "all_lectures_study_module.html"
)

assert RAG_CONFIG_PATH.exists(), (
    "Run the all-lectures Notebook 2 first. Missing: "
    f"{RAG_CONFIG_PATH}"
)

with open(RAG_CONFIG_PATH, "r", encoding="utf-8") as f:
    rag_config = json.load(f)

print("RAG configuration loaded.")
print("Lectures:", rag_config["lecture_numbers"])
print("Collection:", rag_config["collection_name"])


RAG configuration loaded.
Lectures: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34]
Collection: mit_18_06_lectures_01_34


# Step 1: Load Notebook 2's All-Lectures Reranked RAG Resources


In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from sentence_transformers import CrossEncoder

embedding_model = HuggingFaceEmbeddings(
    model_name=rag_config["embedding_model"]
)

vectorstore = Chroma(
    collection_name=rag_config["collection_name"],
    embedding_function=embedding_model,
    persist_directory=rag_config["persist_directory"],
)

reranker = CrossEncoder(
    rag_config["reranker_model"]
)

RETRIEVAL_K = rag_config["retrieval_k"]
RERANK_TOP_N = rag_config["rerank_top_n"]

print("Persistent Chroma loaded")
print("Retrieve:", RETRIEVAL_K)
print("Rerank top N:", RERANK_TOP_N)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Persistent Chroma loaded
Retrieve: 8
Rerank top N: 3


In [ ]:
def seconds_to_timestamp(seconds):
    seconds = int(seconds)

    hours = seconds // 3600
    minutes = (seconds % 3600) // 60
    secs = seconds % 60

    if hours > 0:
        return f"{hours:02d}:{minutes:02d}:{secs:02d}"

    return f"{minutes:02d}:{secs:02d}"


def build_timestamp_link(video_url, seconds):
    return f"{video_url}#t={int(seconds)}"


def retrieve_and_rerank(
    query,
    lecture_number=None,
    candidate_k=RETRIEVAL_K,
    top_n=RERANK_TOP_N,
):
    search_kwargs = {
        "k": candidate_k,
    }

    if lecture_number is not None:
        search_kwargs["filter"] = {
            "lecture_number": int(lecture_number)
        }

    candidates = vectorstore.similarity_search(
        query,
        **search_kwargs,
    )

    if not candidates:
        return []

    pairs = [
        [query, doc.page_content]
        for doc in candidates
    ]

    scores = reranker.predict(pairs)

    ranked = sorted(
        zip(candidates, scores),
        key=lambda item: float(item[1]),
        reverse=True,
    )

    final_docs = []

    for rank, (doc, score) in enumerate(
        ranked[:top_n],
        start=1,
    ):
        doc.metadata["rerank_score"] = float(score)
        doc.metadata["rerank_position"] = rank
        final_docs.append(doc)

    return final_docs


# Step 2: Configure OpenAI + LangSmith


In [ ]:
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
print("OPENAI_API_KEY loaded")

OPENAI_API_KEY loaded


In [ ]:
import os
import langsmith as ls
from google.colab import userdata

langsmith_client = ls.Client(
    api_key=userdata.get("LANGSMITH_API_KEY"),
    api_url="https://eu.api.smith.langchain.com",
    workspace_id=userdata.get("LANGSMITH_WORKSPACE_ID"),
)

os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "students-channel-brain"

print("LangSmith EU client ready")
print("Project: students-channel-brain")

LangSmith EU client ready
Project: students-channel-brain


In [ ]:
print("API key:", bool(os.getenv("LANGSMITH_API_KEY")))
print("Workspace ID:", bool(os.getenv("LANGSMITH_WORKSPACE_ID")))
print("Endpoint:", os.getenv("LANGSMITH_ENDPOINT"))
print("Tracing:", os.getenv("LANGSMITH_TRACING"))
print("Project:", os.getenv("LANGSMITH_PROJECT"))

API key: False
Workspace ID: False
Endpoint: None
Tracing: true
Project: students-channel-brain


In [ ]:
from langchain_openai import ChatOpenAI

LLM_MODEL = "gpt-5.6-luna" # changed from "gpt-4o-mini"

llm = ChatOpenAI(
    model=LLM_MODEL,
    #temperature=0,
    reasoning_effort="high",
    use_responses_api=True,
)

print("LLM ready:", LLM_MODEL)
print("Reasoning effort: high")
print("Responses API: enabled")

LLM ready: gpt-5.6-luna
Reasoning effort: high
Responses API: enabled


# Step 3: Agent State


In [ ]:
from langchain.agents import AgentState


class TutorState(AgentState):
    # Wrong quiz answers with lecture/source metadata.
    weak_topics: NotRequired[
        Annotated[list[dict], operator.add]
    ]

    # One record is appended every time a quiz is graded.
    quiz_results: NotRequired[
        Annotated[list[dict], operator.add]
    ]

    current_quiz: NotRequired[dict | None]

The agent now has two kinds of memory:

```text
Conversation memory
→ previous questions and answers
→ resolves references such as "the two"

Application state
→ current_quiz
→ weak_topics
```


# Step 4: Agent Tools


In [ ]:
from langchain.tools import tool, ToolRuntime
from langchain.messages import ToolMessage
from langgraph.types import Command


@tool
def search_lecture(
    query: str,
    lecture_number: int | None = None,
) -> str:
    """
    Search MIT 18.06 Lectures 1–34 using shared retrieval + reranking.

    If the student explicitly asks about a specific lecture,
    pass lecture_number. Otherwise search all 34 lectures.
    """

    docs = retrieve_and_rerank(
        query,
        lecture_number=lecture_number,
    )

    if not docs:
        scope = (
            f"Lecture {lecture_number}"
            if lecture_number is not None
            else "Lectures 1–34"
        )
        return f"No relevant evidence was found in {scope}."

    parts = []

    for i, doc in enumerate(docs, 1):
        start = doc.metadata["start_seconds"]
        end = doc.metadata["end_seconds"]

        watch_url = build_timestamp_link(
            doc.metadata["video_url"],
            start,
        )

        parts.append(
            f"SOURCE {i}\n"
            f"LECTURE_ID: {doc.metadata['lecture_id']}\n"
            f"LECTURE_NUMBER: {doc.metadata['lecture_number']}\n"
            f"LECTURE_TITLE: {doc.metadata['lecture_title']}\n"
            f"TIMESTAMP: {seconds_to_timestamp(start)}-"
            f"{seconds_to_timestamp(end)}\n"
            f"VIDEO_URL: {doc.metadata['video_url']}\n"
            f"WATCH: {watch_url}\n"
            f"EVIDENCE:\n{doc.page_content}"
        )

    return "\n\n".join(parts)


In [ ]:
from pydantic import BaseModel, Field


class QuizItem(BaseModel):
    question: str
    options: list[str] = Field(min_length=4, max_length=4)
    correct_answer: Literal["A", "B", "C", "D"]
    topic: str
    explanation: str
    source_timestamp: str


quiz_llm = llm.with_structured_output(QuizItem)

In [ ]:
from pydantic import BaseModel


class TopicSupportDecision(BaseModel):
    supported: bool
    reason: str


topic_support_llm = llm.with_structured_output(
    TopicSupportDecision
)


def check_topic_supported_by_lecture(
    topic,
    docs,
):
    """
    Decide whether the requested topic is genuinely supported
    by the retrieved evidence from Lectures 1–34.

    Vector retrieval always returns nearest chunks, so this gate
    prevents unrelated topics from being forced into course content.
    """

    clean_topic = (topic or "").strip()

    if not clean_topic or not docs:
        return TopicSupportDecision(
            supported=False,
            reason="No usable topic or course evidence was found.",
        )

    evidence = "\n\n".join(
        (
            f"Lecture {d.metadata['lecture_number']} — "
            f"{d.metadata['lecture_title']}\n"
            f"Timestamp: "
            f"{seconds_to_timestamp(d.metadata['start_seconds'])}-"
            f"{seconds_to_timestamp(d.metadata['end_seconds'])}\n"
            f"{d.page_content}"
        )
        for d in docs[:5]
    )

    return topic_support_llm.invoke(
        f"""
You are a strict scope checker for
MIT 18.06 Linear Algebra Lectures 1–34.

Requested topic:
{clean_topic}

Retrieved course evidence:
{evidence}

Decide whether the requested topic itself is genuinely taught or
meaningfully discussed in this evidence.

Rules:
- Return supported=true only when the evidence clearly supports
  teaching or studying the requested topic.
- Do not mark a topic supported merely because vector search returned
  nearest-neighbor chunks.
- If the topic belongs to an unrelated subject, return supported=false.
- If only generic words overlap but the underlying concept is unrelated,
  return supported=false.
- Be strict: when uncertain, return supported=false.
- Give a short reason.
"""
    )


In [ ]:
@tool
def create_quiz(
    topic: str,
    runtime: ToolRuntime,
    lecture_number: int | None = None,
) -> Command:
    """
    Create one grounded four-option multiple-choice quiz.

    Use lecture_number when the student explicitly requests
    a quiz from a specific lecture. Otherwise search Lectures 1–34.

    Every new quiz request replaces the previous active quiz,
    even if the previous quiz was not answered.
    """

    clean_topic = (topic or "").strip()

    docs = retrieve_and_rerank(
        clean_topic,
        lecture_number=lecture_number,
    )

    if not docs:
        scope = (
            f"Lecture {lecture_number}"
            if lecture_number is not None
            else "Lectures 1–34"
        )

        return Command(
            update={
                "current_quiz": None,
                "messages": [
                    ToolMessage(
                        content=(
                            f'I could not find "{clean_topic}" '
                            f"in {scope}."
                        ),
                        tool_call_id=runtime.tool_call_id,
                    )
                ],
            }
        )

    support = check_topic_supported_by_lecture(
        clean_topic,
        docs,
    )

    if not support.supported:
        return Command(
            update={
                "current_quiz": None,
                "messages": [
                    ToolMessage(
                        content=(
                            "### Topic not covered\n\n"
                            f'"{clean_topic}" does not appear to be '
                            "covered in the available MIT 18.06 "
                            "Lectures 1–34 evidence."
                        ),
                        tool_call_id=runtime.tool_call_id,
                    )
                ],
            }
        )

    context = "\n\n".join(
        (
            f"Lecture {d.metadata['lecture_number']} — "
            f"{d.metadata['lecture_title']} | "
            f"{seconds_to_timestamp(d.metadata['start_seconds'])}-"
            f"{seconds_to_timestamp(d.metadata['end_seconds'])}: "
            f"{d.page_content}"
        )
        for d in docs
    )

    previous_quiz = runtime.state.get(
        "current_quiz"
    )

    previous_question = (
        previous_quiz.get("question")
        if previous_quiz
        else None
    )

    quiz = quiz_llm.invoke(
        f"""
Create ONE multiple-choice question using only
the supplied MIT 18.06 Lectures 1–34 evidence.

Requested topic:
{clean_topic}

Evidence:
{context}

Previous quiz question:
{previous_question if previous_question else "None"}

Requirements:
- the question MUST directly test the requested topic;
- exactly four options;
- exactly one correct answer;
- options must contain answer text only;
- do NOT include option letters inside option text;
- plausible Linear Algebra distractors;
- explanation says why the correct answer is correct;
- do not reveal the answer in the question;
- if a previous quiz exists, create a different question;
- do not repeat the same wording;
- test a different aspect when the evidence allows it;
- the topic should be specific and concise:
- never use broad labels such as "Linear Algebra",
  "Mathematics", "Lecture 1", or "Matrices".
"""
    )

    quiz_data = quiz.model_dump()
    quiz_data["topic"] = clean_topic

    quiz_data["options"] = [
        re.sub(
            r"^\s*[ABCDabcd][\.\:\)]\s*",
            "",
            text,
        )
        for text in quiz_data["options"]
    ]

    # --------------------------------------------------------
    # Randomize answer positions.
    #
    # The LLM may prefer putting the correct answer first.
    # Shuffle the four generated options after generation,
    # then update correct_answer to the new position.
    # This avoids a fixed answer pattern.
    # --------------------------------------------------------
    import random

    letters = ["A", "B", "C", "D"]

    original_correct = quiz_data["correct_answer"].upper()
    original_correct_index = letters.index(original_correct)

    paired_options = [
        (option, i == original_correct_index)
        for i, option in enumerate(quiz_data["options"])
    ]

    random.shuffle(paired_options)

    quiz_data["options"] = [
        option
        for option, _ in paired_options
    ]

    new_correct_index = next(
        i
        for i, (_, is_correct) in enumerate(paired_options)
        if is_correct
    )

    quiz_data["correct_answer"] = letters[new_correct_index]

    # Ground source metadata comes from retrieval, not the LLM.
    primary = docs[0]
    start = primary.metadata["start_seconds"]
    end = primary.metadata["end_seconds"]

    quiz_data.update({
        "lecture_id": primary.metadata["lecture_id"],
        "lecture_number": primary.metadata["lecture_number"],
        "lecture_title": primary.metadata["lecture_title"],
        "video_url": primary.metadata["video_url"],
        "timestamp": (
            f"{seconds_to_timestamp(start)}-"
            f"{seconds_to_timestamp(end)}"
        ),
        "video_link": build_timestamp_link(
            primary.metadata["video_url"],
            start,
        ),
    })

    option_lines = "\n\n".join(
        f"{letter}. {option}"
        for letter, option in zip(
            ["A", "B", "C", "D"],
            quiz_data["options"],
        )
    )

    return Command(
        update={
            "current_quiz": quiz_data,
            "messages": [
                ToolMessage(
                    content=(
                        "### Quiz\n\n"
                        f"{quiz_data['question']}\n\n"
                        f"{option_lines}\n\n"
                        "Choose A, B, C, or D."
                    ),
                    tool_call_id=runtime.tool_call_id,
                )
            ],
        }
    )

In [ ]:
@tool
def grade_quiz_answer(
    student_answer: str,
    runtime: ToolRuntime,
) -> Command:
    """Grade the active quiz and record performance/source metadata."""

    quiz = runtime.state.get("current_quiz")

    if not quiz:
        return Command(
            update={
                "messages": [
                    ToolMessage(
                        content="There is no active quiz to grade.",
                        tool_call_id=runtime.tool_call_id,
                    )
                ]
            }
        )

    match = re.match(
        r"\s*([ABCDabcd])",
        student_answer or "",
    )

    if not match:
        return Command(
            update={
                "messages": [
                    ToolMessage(
                        content="Please answer with A, B, C, or D.",
                        tool_call_id=runtime.tool_call_id,
                    )
                ]
            }
        )

    selected = match.group(1).upper()
    correct = quiz["correct_answer"].upper()

    correct_index = [
        "A", "B", "C", "D"
    ].index(correct)

    correct_text = quiz["options"][correct_index]
    is_correct = selected == correct

    result_record = {
        "topic": quiz["topic"],
        "correct": is_correct,
        "lecture_id": quiz.get("lecture_id"),
        "lecture_number": quiz.get("lecture_number"),
        "lecture_title": quiz.get("lecture_title"),
        "timestamp": quiz.get("timestamp"),
        "video_url": quiz.get("video_url"),
        "video_link": quiz.get("video_link"),
    }

    updates = {
        "current_quiz": None,
        "quiz_results": [result_record],
    }

    if is_correct:
        feedback = (
            "### You're right.\n\n"
            f"That's because {quiz['explanation']}"
        )
    else:
        updates["weak_topics"] = [result_record]

        feedback = (
            "### You're wrong.\n\n"
            f"That's because {quiz['explanation']}\n\n"
            f"The correct answer is {correct}. {correct_text}."
        )

    updates["messages"] = [
        ToolMessage(
            content=feedback,
            tool_call_id=runtime.tool_call_id,
        )
    ]

    return Command(update=updates)


In [ ]:
@tool
def get_weak_topics(runtime: ToolRuntime) -> str:
    """Return weak topics recorded for the current student thread."""

    topics = runtime.state.get(
        "weak_topics",
        [],
    )

    if not topics:
        return "No weak topics recorded yet."

    unique = {}

    for item in topics:
        key = (
            item.get("topic"),
            item.get("lecture_id"),
        )
        unique[key] = item

    lines = ["### Topics to Review"]

    for item in unique.values():
        line = (
            f"- **{item['topic']}** | "
            f"Lecture {item.get('lecture_number')}: "
            f"{item.get('lecture_title')} | "
            f"{item.get('timestamp')}"
        )

        if item.get("video_link"):
            line += f" | [Watch]({item['video_link']})"

        lines.append(line)

    return "\n".join(lines)


# Step 5: Build the Conversational Tutor Agent


In [ ]:
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents import create_agent

checkpointer = InMemorySaver()

SYSTEM_PROMPT = """
You are a Tutor Agent for MIT 18.06 Linear Algebra Lectures 1–34.

MEMORY
- Use the full conversation history in the current thread.
- Resolve follow-ups such as "the two", "both", "that", and "compare them".
- When a follow-up refers to earlier concepts, call search_lecture with a
  standalone query that explicitly names those concepts.

SCOPE
- Only answer questions related to MIT 18.06 Linear Algebra Lectures 1–34
  or the student's course study activity.
- For clearly unrelated questions, do not call course tools and do not answer
  from general knowledge.
- Reply only:
  "That question is outside the available MIT 18.06 Linear Algebra course material."

COURSE ANSWERS
- For factual or conceptual course questions, ALWAYS call search_lecture.
- If a lecture number is explicitly named, pass that lecture_number.
- Otherwise search Lectures 1–34.
- Base the answer only on retrieved course evidence.

FORMAT
- Start with a short level-2 Markdown heading.
- Be concise and student-friendly.
- Use standard Markdown bullets when useful.
- Use backticks for short inline symbols such as `A`, `x`, `b`, and `Ax = b`.
- Do not use inline LaTeX.
- Put important equations on separate lines using matching $$ ... $$ delimiters.
- Keep equations simple.

SOURCE
End each grounded course answer with:

### Source

For each materially supporting source use:

- **Lecture N: Lecture Title** MM:SS–MM:SS
  - [Watch this part](WATCH_URL)

Use only retrieved lecture metadata, timestamps, and WATCH URLs.
Do not list irrelevant retrieved chunks.

QUIZZES
- If the student asks for a quiz, ALWAYS call create_quiz.
- If a lecture number is explicitly requested, pass that lecture_number.
- A new quiz replaces the previous active quiz.
- If an active quiz exists and the student answers A/B/C/D,
  ALWAYS call grade_quiz_answer.
- Do not grade the quiz yourself.

WEAK TOPICS
- Wrong quiz answers are stored with topic and lecture/source metadata.
- If the student asks what they are struggling with,
  ALWAYS call get_weak_topics.
"""

agent = create_agent(
    model=llm,
    tools=[
        search_lecture,
        create_quiz,
        grade_quiz_answer,
        get_weak_topics,
    ],
    system_prompt=SYSTEM_PROMPT,
    checkpointer=checkpointer,
    state_schema=TutorState,
)

print("Multi-lecture Conversational Tutor Agent ready")

Multi-lecture Conversational Tutor Agent ready


# Step 6: Website Session Helpers


In [ ]:
def initial_web_state():
    return {
        "thread_id": (
            "web-"
            + str(uuid.uuid4())
        ),
        "last_course_topic": None,
    }


def agent_config_from_state(state):
    state = state or initial_web_state()

    return {
        "configurable": {
            "thread_id": state["thread_id"]
        }
    }


def agent_snapshot_values(state):
    config = agent_config_from_state(state)
    snapshot = agent.get_state(config)
    return snapshot.values


def weak_topics_from_agent(state):
    values = agent_snapshot_values(state)

    topics = values.get(
        "weak_topics",
        [],
    )

    unique = {}

    for item in topics:
        key = (
            item.get("topic"),
            item.get("lecture_id"),
        )
        unique[key] = item

    return list(unique.values())


def quiz_results_from_agent(state):
    values = agent_snapshot_values(state)

    return values.get(
        "quiz_results",
        [],
    )


def topic_quiz_stats(topic_record, state):
    results = quiz_results_from_agent(state)

    topic_key = str(
        topic_record.get("topic", "")
    ).strip().lower()

    lecture_id = topic_record.get("lecture_id")

    matching = [
        item
        for item in results
        if (
            str(item.get("topic", "")).strip().lower()
            == topic_key
            and item.get("lecture_id") == lecture_id
        )
    ]

    attempts = len(matching)

    correct = sum(
        1
        for item in matching
        if item.get("correct") is True
    )

    success_rate = (
        round(100 * correct / attempts)
        if attempts
        else 0
    )

    return {
        "attempts": attempts,
        "correct": correct,
        "success_rate": success_rate,
    }


def weak_topics_markdown(state):
    topics = weak_topics_from_agent(state)

    if not topics:
        return (
            "## Topics to Review\n\n"
            "No weak topics recorded yet. "
            "Complete a quiz to start tracking your progress."
        )

    sections = [
        "## Topics to Review"
    ]

    for item in topics:
        stats = topic_quiz_stats(
            item,
            state,
        )

        section = [
            f"### {item['topic']}",
            (
                "- **Lecture:** "
                f"{item.get('lecture_number')} — "
                f"{item.get('lecture_title')}"
            ),
            (
                "- **Success rate:** "
                f"{stats['correct']}/{stats['attempts']} correct "
                f"({stats['success_rate']}%)"
            ),
            (
                "- **Timestamp:** "
                f"{item.get('timestamp')}"
            ),
        ]

        if item.get("video_link"):
            section.append(
                "- [Watch this part]"
                f"({item['video_link']})"
            )

        sections.append(
            "\n".join(section)
        )

    return "\n\n".join(sections)


# Step 7: Website Functions


In [ ]:
def extract_text(content):
    """
    Convert Chat Completions or Responses API content
    into plain Markdown text for Gradio.
    """

    if isinstance(content, str):
        return content

    if isinstance(content, list):
        text_parts = []

        for item in content:
            if not isinstance(item, dict):
                continue

            if item.get("type") in {
                "text",
                "output_text",
            }:
                text = item.get("text")

                if isinstance(text, str) and text.strip():
                    text_parts.append(text)

        return "\n\n".join(text_parts)

    return str(content)


OUT_OF_SCOPE_MESSAGE = (
    "That question is outside the available MIT 18.06 "
    "Linear Algebra course material."
)


def web_agent_chat(
    message,
    history,
    state,
):
    state = state or initial_web_state()

    clean_message = (message or "").strip()
    history = list(history or [])

    if not clean_message:
        return history, "", state

    with ls.tracing_context(
        client=langsmith_client,
        project_name="students-channel-brain",
        enabled=True,
    ):
        result = agent.invoke(
            {
                "messages": [
                    {
                        "role": "user",
                        "content": clean_message,
                    }
                ]
            },
            agent_config_from_state(state),
        )

    answer = extract_text(
        result["messages"][-1].content
    )

    answer = clean_gradio_markdown(
        answer
    )

    # Only valid course questions can become the Course Builder fallback topic.
    if OUT_OF_SCOPE_MESSAGE not in answer:
        state["last_course_topic"] = clean_message

    # Always return the Tutor turn, including out-of-scope refusals.
    history.extend(
        [
            {
                "role": "user",
                "content": clean_message,
            },
            {
                "role": "assistant",
                "content": answer,
            },
        ]
    )

    return (
        history,
        "",
        state,
    )

In [ ]:
import re


def clean_gradio_markdown(text: str) -> str:
    """
    Normalize model output for Gradio Chatbot rendering.
    """

    if not isinstance(text, str):
        text = str(text)

    # Fix escaped dollar signs if the model still produces them
    text = text.replace(r"\$", "$")

    # Convert common non-Markdown bullet characters
    text = re.sub(
        r"(?m)^\s*[○◦•]\s+",
        "- ",
        text,
    )

    # Add clean spacing around headings
    text = re.sub(
        r"(?m)^(#{1,6}\s+.+)$",
        r"\n\1\n",
        text,
    )

    # Reduce excessive blank lines
    text = re.sub(
        r"\n{3,}",
        "\n\n",
        text,
    )

    return text.strip()

In [ ]:
def web_create_quiz(
    topic,
    state,
):
    import gradio as gr

    state = state or initial_web_state()
    clean_topic = (topic or "").strip() or "linear equations"
    config = agent_config_from_state(state)

    with ls.tracing_context(
        client=langsmith_client,
        project_name="students-channel-brain",
        enabled=True,
    ):
        result = agent.invoke(
            {
                "messages": [
                    {
                        "role": "user",
                        "content": (
                            f"Quiz me on {clean_topic}. "
                            "Create one multiple-choice question."
                        ),
                    }
                ]
            },
            config,
        )

    snapshot = agent.get_state(config)
    quiz = snapshot.values.get("current_quiz")

    if not quiz:
        refusal_message = extract_text(
            result["messages"][-1].content
        )

        return (
            refusal_message,
            gr.update(
                choices=["A", "B", "C", "D"],
                value=None,
                visible=False,
            ),
            gr.update(visible=False),
            state,
        )

    letters = ["A", "B", "C", "D"]

    options = "\n\n".join(
        f"{letter}. {text}"
        for letter, text in zip(
            letters,
            quiz["options"],
        )
    )

    quiz_markdown = (
        "## Quiz\n\n"
        f"{quiz['question']}\n\n"
        f"{options}\n\n"
        f"**Source:** Lecture {quiz.get('lecture_number')} — "
        f"{quiz.get('lecture_title')} | "
        f"{quiz.get('timestamp')}"
    )

    return (
        quiz_markdown,
        gr.update(
            choices=letters,
            value=None,
            visible=True,
        ),
        gr.update(visible=True),
        state,
    )


def web_grade_quiz(
    selected_answer,
    state,
):
    state = state or initial_web_state()

    if not selected_answer:
        return (
            "Choose A, B, C, or D.",
            state,
        )

    with ls.tracing_context(
        client=langsmith_client,
        project_name="students-channel-brain",
        enabled=True,
    ):
        result = agent.invoke(
            {
                "messages": [
                    {
                        "role": "user",
                        "content": f"My quiz answer is {selected_answer}",
                    }
                ]
            },
            agent_config_from_state(state),
        )

    feedback = extract_text(
    result["messages"][-1].content
    )

    return (
        feedback,
        state,
    )


def refresh_weak_topics(state):
    return weak_topics_markdown(state)

# Step 8: Course Builder


In [ ]:
class StudyModule(BaseModel):
    title: str
    learning_objectives: list[str]
    explanation: str
    key_points: list[str]
    worked_example: str


course_llm = llm.with_structured_output(StudyModule)

In [ ]:
def build_study_module(
    topic,
    lecture_number=None,
):
    clean_topic = (topic or "").strip()

    docs = retrieve_and_rerank(
        clean_topic,
        lecture_number=lecture_number,
        candidate_k=RETRIEVAL_K,
        top_n=RERANK_TOP_N,
    )

    if not docs:
        scope = (
            f"Lecture {lecture_number}"
            if lecture_number is not None
            else "Lectures 1–34"
        )

        raise ValueError(
            f'"{clean_topic}" was not found in {scope}.'
        )

    support = check_topic_supported_by_lecture(
        clean_topic,
        docs,
    )

    if not support.supported:
        raise ValueError(
            f'"{clean_topic}" does not appear to be covered '
            "in the available MIT 18.06 Lectures 1–34 evidence."
        )

    context = "\n\n".join(
        (
            f"SOURCE {i}\n"
            f"Lecture {d.metadata['lecture_number']} — "
            f"{d.metadata['lecture_title']}\n"
            f"Timestamp: "
            f"{seconds_to_timestamp(d.metadata['start_seconds'])}-"
            f"{seconds_to_timestamp(d.metadata['end_seconds'])}\n"
            f"{d.page_content}"
        )
        for i, d in enumerate(docs, 1)
    )

    module = course_llm.invoke(
        f"""
Create a compact self-study module for
MIT 18.06 Linear Algebra Lectures 1–34.

Topic:
{clean_topic}

Use ONLY this course evidence:
{context}

Requirements:
- the module MUST directly teach the requested topic;
- 2 to 4 learning objectives;
- clear explanation;
- 3 to 6 key points;
- one conceptual worked example;
- concise and student-friendly language;
- do not include a quiz;
- do not invent material outside the evidence.
"""
    )

    sources = []

    for d in docs:
        start = d.metadata["start_seconds"]
        end = d.metadata["end_seconds"]

        sources.append(
            {
                "lecture_id": d.metadata["lecture_id"],
                "lecture_number": d.metadata["lecture_number"],
                "lecture_title": d.metadata["lecture_title"],
                "timestamp": (
                    f"{seconds_to_timestamp(start)}-"
                    f"{seconds_to_timestamp(end)}"
                ),
                "video_url": d.metadata["video_url"],
                "video_link": build_timestamp_link(
                    d.metadata["video_url"],
                    start,
                ),
            }
        )

    return module, sources

In [ ]:
from html import escape


def render_course_html(
    topic,
    module,
    sources,
):
    objectives_html = "\n".join(
        f"<li>{escape(item)}</li>"
        for item in module.learning_objectives
    )

    key_points_html = "\n".join(
        f"<li>{escape(item)}</li>"
        for item in module.key_points
    )

    source_links_html = "\n".join(
        (
            f'<li><a href="{escape(source["video_link"])}" '
            f'target="_blank">'
            f'Lecture {source["lecture_number"]} — '
            f'{escape(source["lecture_title"])} — '
            f'{escape(source["timestamp"])}</a></li>'
        )
        for source in sources
    )

    html_content = f"""<!doctype html>
<html>
<head>
<meta charset="utf-8">
<title>{escape(module.title)}</title>

<style>
body {{
    font-family: Arial, sans-serif;
    max-width: 900px;
    margin: 40px auto;
    line-height: 1.6;
    padding: 0 20px;
}}

h1, h2 {{
    color: #A31F34;
}}

section {{
    margin-bottom: 28px;
}}

.card {{
    border: 1px solid #ddd;
    border-radius: 10px;
    padding: 18px;
}}
</style>
</head>

<body>

<h1>{escape(module.title)}</h1>

<p>
<strong>Course:</strong>
MIT 18.06 Linear Algebra — Lectures 1–34
</p>

<p>
<strong>Study topic:</strong>
{escape(topic)}
</p>

<section>
<h2>Learning Objectives</h2>
<ul>
{objectives_html}
</ul>
</section>

<section>
<h2>Explanation</h2>
<p>
{escape(module.explanation)}
</p>
</section>

<section>
<h2>Key Points</h2>
<ul>
{key_points_html}
</ul>
</section>

<section>
<h2>Worked Example</h2>

<div class="card">
{escape(module.worked_example)}
</div>

</section>

<section>
<h2>Watch Relevant Lecture Sections</h2>

<ul>
{source_links_html}
</ul>

</section>

</body>
</html>
"""

    HTML_OUTPUT_PATH.write_text(
        html_content,
        encoding="utf-8",
    )

    return html_content


In [ ]:
def web_build_course(
    topic,
    state,
):
    import gradio as gr

    state = state or initial_web_state()

    clean_topic = (topic or "").strip()
    lecture_number = None

    fallback_message = None

    if not clean_topic:
        weak_topics = weak_topics_from_agent(state)

        if weak_topics:
            first_weak = weak_topics[0]

            clean_topic = first_weak["topic"]

            lecture_number = first_weak.get(
                "lecture_number"
            )

            fallback_message = (
                "No topic was entered, so this module was created "
                f"from your first recorded weak topic: "
                f"**{clean_topic}**."
            )

        else:
            recent_topic = (
                state.get("last_course_topic")
                or ""
            ).strip()

            if recent_topic:
                clean_topic = recent_topic

                fallback_message = (
                    "No weak topics have been recorded yet, "
                    "so this module was created from your most recent "
                    f"MIT 18.06 topic: **{clean_topic}**."
                )

            else:
                return (
                    (
                        "No weak topics have been recorded yet, and "
                        "there is no recent MIT 18.06 topic to use. "
                        "Ask a Tutor question or enter a study topic first."
                    ),
                    "",
                    gr.update(
                        value="",
                        visible=False,
                    ),
                )

    try:
        module, sources = build_study_module(
            clean_topic,
            lecture_number=lecture_number,
        )

        html_content = render_course_html(
            clean_topic,
            module,
            sources,
        )

        summary_md = (
    (
        f"> {fallback_message}\n\n"
        if fallback_message
        else ""
    )
    + f"### {module.title}\n\n"
    + f"{module.explanation}\n\n"
    + "Your personalized study module is ready."
)

        file_url = (
            "/gradio_api/file="
            + str(HTML_OUTPUT_PATH)
        )

        link_html = f"""
<div style="margin-top:18px;">
<a
    href="{file_url}"
    target="_blank"
    style="
        display:inline-block;
        background:#A31F34;
        color:white;
        padding:12px 20px;
        border-radius:8px;
        text-decoration:none;
        font-weight:600;
    "
>
Open Study Module in New Tab
</a>
</div>
"""

        return (
            summary_md,
            html_content,
            gr.update(
                value=link_html,
                visible=True,
            ),
        )

    except Exception as e:
        return (
            f"Could not build study module: {e}",
            "",
            gr.update(
                value="",
                visible=False,
            ),
        )


# Step 9: Build the Gradio Website


In [ ]:
def course_toc_markdown():
    """
    Build Course Contents dynamically from lecture metadata
    already stored in the full-course Chroma index.
    """
    data = vectorstore.get(include=["metadatas"])

    lecture_map = {}

    for metadata in data.get("metadatas", []):
        if not metadata:
            continue

        lecture_number = metadata.get("lecture_number")
        lecture_title = metadata.get("lecture_title")

        if lecture_number is None:
            continue

        lecture_number = int(lecture_number)

        if 1 <= lecture_number <= 34:
            lecture_map[lecture_number] = (
                str(lecture_title).strip()
                if lecture_title
                else f"Lecture {lecture_number}"
            )

    missing = [
        number
        for number in range(1, 35)
        if number not in lecture_map
    ]

    if missing:
        return (
            "### Course Contents\n\n"
            "Course metadata is incomplete. Missing lectures: "
            + ", ".join(map(str, missing))
        )

    lines = [
        "### Course Contents",
        "",
        "MIT 18.06 Linear Algebra — Lectures 1–34",
        "",
    ]

    for lecture_number in range(1, 35):
        lines.append(
            f"- **Lecture {lecture_number}:** "
            f"{lecture_map[lecture_number]}"
        )

    return "\n".join(lines)


COURSE_TOC = course_toc_markdown()

print(COURSE_TOC)


### Course Contents

MIT 18.06 Linear Algebra — Lectures 1–34

- **Lecture 1:** The geometry of linear equations
- **Lecture 2:** Elimination with matrices
- **Lecture 3:** Multiplication and inverse matrices
- **Lecture 4:** Factorization into A = LU
- **Lecture 5:** Transposes, permutations, spaces R^n
- **Lecture 6:** Column space and nullspace
- **Lecture 7:** Solving Ax = 0: pivot variables, special solutions
- **Lecture 8:** Solving Ax = b: row reduced form R
- **Lecture 9:** Independence, basis, and dimension
- **Lecture 10:** The four fundamental subspaces
- **Lecture 11:** Matrix spaces; rank 1; small world graphs
- **Lecture 12:** Graphs, networks, incidence matrices
- **Lecture 13:** Quiz 1 review
- **Lecture 14:** Orthogonal vectors and subspaces
- **Lecture 15:** Projections onto subspaces
- **Lecture 16:** Projection matrices and least squares
- **Lecture 17:** Orthogonal matrices and Gram-Schmidt
- **Lecture 18:** Properties of determinants
- **Lecture 19:** Determinant 

In [ ]:
import gradio as gr

custom_css = """
.gradio-container {
    width: 95% !important;
    max-width: 95% !important;
    margin: 0 auto !important;
}

button[role="tab"][aria-selected="true"] {
    color: #A31F34 !important;
    border-bottom-color: #A31F34 !important;
}

button.primary {
    background: #A31F34 !important;
    border-color: #A31F34 !important;
    color: white !important;
    font-weight: 700 !important;
    min-height: 54px !important;
    font-size: 18px !important;
}

button.primary:hover {
    background: #86192B !important;
    border-color: #86192B !important;
}

#tutor-chat {
    width: 100% !important;
    min-height: 520px !important;
}

#tutor-chat .prose,
#weak-topics-output .prose,
#quiz-output .prose {
    font-size: 16px !important;
    line-height: 1.65 !important;
}

textarea,
input {
    font-size: 16px !important;
}
"""


with gr.Blocks(
    title="MIT Linear Algebra Tutor — Lectures 1–34"
) as demo:

    session_state = gr.State(
        initial_web_state()
    )

    gr.Markdown(
        "# MIT Linear Algebra Tutor\n"
        "MIT 18.06 — Lectures 1–34"
    )

    with gr.Tab("Tutor"):

        with gr.Accordion(
            "Course Contents — Lectures 1–34",
            open=False,
        ):
            gr.Markdown(COURSE_TOC)

        tutor_chat = gr.Chatbot(
            label="Tutor Conversation",
            height=520,
            elem_id="tutor-chat",
        )

        tutor_question = gr.Textbox(
            label="Message",
            placeholder=(
                "Ask about Lectures 1–34, or specify a lecture number..."
            ),
        )

        tutor_button = gr.Button(
            "Send",
            variant="primary",
        )

        tutor_button.click(
            fn=web_agent_chat,
            inputs=[
                tutor_question,
                tutor_chat,
                session_state,
            ],
            outputs=[
                tutor_chat,
                tutor_question,
                session_state,
            ],
        )

        tutor_question.submit(
            fn=web_agent_chat,
            inputs=[
                tutor_question,
                tutor_chat,
                session_state,
            ],
            outputs=[
                tutor_chat,
                tutor_question,
                session_state,
            ],
        )


    with gr.Tab("Quiz"):

        quiz_topic = gr.Textbox(
            label="Quiz topic",
            placeholder=(
                "Example: LU factorization from Lecture 4"
            ),
        )

        quiz_button = gr.Button(
            "Create Quiz",
            variant="primary",
        )

        quiz_output = gr.Markdown(
            elem_id="quiz-output"
        )

        quiz_answer = gr.Radio(
            choices=["A", "B", "C", "D"],
            label="Your answer",
            visible=False,
        )

        grade_button = gr.Button(
            "Submit Answer",
            variant="primary",
            visible=False,
        )

        quiz_feedback = gr.Markdown()

        quiz_button.click(
            fn=web_create_quiz,
            inputs=[
                quiz_topic,
                session_state,
            ],
            outputs=[
                quiz_output,
                quiz_answer,
                grade_button,
                session_state,
            ],
        )

        grade_button.click(
            fn=web_grade_quiz,
            inputs=[
                quiz_answer,
                session_state,
            ],
            outputs=[
                quiz_feedback,
                session_state,
            ],
        )

    with gr.Tab("Weak Topics"):

        weak_refresh = gr.Button(
            "Refresh Weak Topics",
            variant="primary",
        )

        weak_output = gr.Markdown(
            elem_id="weak-topics-output"
        )

        weak_refresh.click(
            fn=refresh_weak_topics,
            inputs=session_state,
            outputs=weak_output,
        )

    with gr.Tab("Course Builder"):

        course_topic = gr.Textbox(
            label="Study topic",
            placeholder=(
    "Leave blank to use your first weak topic "
    "or most recent course topic"
),
        )

        build_button = gr.Button(
            "Build Study Module",
            variant="primary",
        )

        course_summary = gr.Markdown()
        course_html = gr.HTML()

        course_link = gr.HTML(
            value="",
            visible=False,
        )

        build_button.click(
            fn=web_build_course,
            inputs=[
                course_topic,
                session_state,
            ],
            outputs=[
                course_summary,
                course_html,
                course_link,
            ],
        )

demo

Gradio Blocks instance: 6 backend functions
-------------------------------------------
fn_index=0
 inputs:
 |-<gradio.components.textbox.Textbox object at 0x7dea4186e900>
 |-<gradio.components.chatbot.Chatbot object at 0x7dea4186e7b0>
 |-<gradio.components.state.State object at 0x7dea4186e270>
 outputs:
 |-<gradio.components.chatbot.Chatbot object at 0x7dea4186e7b0>
 |-<gradio.components.textbox.Textbox object at 0x7dea4186e900>
 |-<gradio.components.state.State object at 0x7dea4186e270>
fn_index=1
 inputs:
 |-<gradio.components.textbox.Textbox object at 0x7dea4186e900>
 |-<gradio.components.chatbot.Chatbot object at 0x7dea4186e7b0>
 |-<gradio.components.state.State object at 0x7dea4186e270>
 outputs:
 |-<gradio.components.chatbot.Chatbot object at 0x7dea4186e7b0>
 |-<gradio.components.textbox.Textbox object at 0x7dea4186e900>
 |-<gradio.components.state.State object at 0x7dea4186e270>
fn_index=2
 inputs:
 |-<gradio.components.textbox.Textbox object at 0x7dea417efc50>
 |-<gradio.compo

# Step 10: Launch the Website


In [ ]:
def launch_app():
    """
    Start a fresh Gradio session.

    Important:
    A Gradio share link runs inside the current Colab runtime.
    If Colab disconnects, that old link cannot stay alive.

    After reconnecting:
    1. rerun the notebook cells needed to rebuild `demo`;
    2. run `launch_app()` again;
    3. use the NEW Gradio link.
    """

    try:
        demo.close()
    except Exception:
        pass

    print(
        "Launching a fresh Gradio session.\n"
        "If Colab disconnected, the old share link is no longer valid; "
        "use the new link printed below."
    )

    return demo.launch(
        share=True,
        debug=False,
        inline=True,
        css=custom_css,
        allowed_paths=[
            str(
                OUTPUT_DIR.resolve()
            )
        ],
    )


launch_app()

Launching a fresh Gradio session.
If Colab disconnected, the old share link is no longer valid; use the new link printed below.
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://5c2887f0bcb37fb2fe.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


# Acceptance Test — Lectures 1–34

Run this sequence from a fresh website session.

## 1. Cross-lecture Tutor

```text
What is the column picture?
In Lecture 4, why does A = LU matter?
```

Pass:
- first question can search the available course;
- second question filters to Lecture 4;
- sources show the correct lecture, timestamp, and video link.

## 2. Conversational Memory

```text
What is the column picture?
What is the row picture?
What are the differences between the two?
```

Pass: the third answer correctly interprets **"the two"**.

## 3. Quiz + Weak Topics

Create a quiz about a topic from one of Lectures 1–34 and answer incorrectly.

Pass:
- weak topic appears;
- lecture number/title appear;
- success rate appears;
- timestamp and watch link appear.

## 4. Course Builder

Leave Study Topic blank after creating a weak topic.

Pass:
- the weak topic is used automatically;
- its lecture is preserved when building the study module;
- HTML is generated;
- relevant lecture links appear.

## 5. LangSmith

Verify that Tutor, Quiz, grading, and Course Builder calls create traces.

Once these pass, the 5-lecture application is ready for the next validation stage.


## 6. Colab / Gradio reconnect check

A Gradio share link depends on the active Colab runtime.

If the runtime disconnects:
- reconnect Colab;
- rerun the notebook cells required to rebuild the app;
- run the final `launch_app()` cell again;
- use the new Gradio share link.

Do not continue using the old share link.

## 7. Quiz answer-position check

Create at least four quizzes in the same website session.

Expected correct-answer positions:

```text
B → C → D → A
```

Then the cycle repeats.

This prevents the structured LLM from making **A** the correct answer most of the time.

## 7. Quiz answer-position check

Create several quizzes.

Expected:
- the correct answer should not consistently be A;
- answer positions should vary naturally because the four generated options
  are shuffled randomly after quiz generation;
- there is no fixed A/B/C/D rotation pattern.

# Scope Guard Verification — Lectures 1–34

The Quiz and Course Builder validate whether the requested topic is
actually supported by the retrieved MIT 18.06 Lectures 1-34 evidence.

## In-scope examples

Use topics that you know occur in the first five lectures, for example:

```text
column picture
row picture
elimination
inverse matrices
LU factorization
```

## Out-of-scope tests

These should be rejected:

```text
US History
photosynthesis
World War II
Python programming
French Revolution
```

Expected behavior:

- **Tutor:** grounded course answers use retrieved course evidence.
- **Quiz:** unsupported topics do not create a quiz.
- **Course Builder:** unsupported topics do not generate a study module.
- Valid sources retain lecture ID/number/title, timestamp, and video link.
